In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-06-butina-clustering.py"

# 相似度分群怎麼選代表分子？
使用 MoleculeNet BACE；第一次執行需連網下載 bace.csv。執行下格後，最後一格會在獨立的 Python 程序計算並畫圖。小練習可以接在寫入程式的儲存格末尾，再重跑該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import numpy as np
import deepchem as dc
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
tasks, (dataset,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
assert tasks == ["pIC50"]
smiles = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in dataset.ids]
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
from rdkit.ML.Cluster import Butina
from rdkit.Chem import Draw

library = sorted(set(smiles))
mols = [Chem.MolFromSmiles(s) for s in library]
fps = [generator.GetFingerprint(m) for m in mols]
distances = []
for i, fp in enumerate(fps):
    similarities = DataStructs.BulkTanimotoSimilarity(fp, fps[:i])
    distances.extend(1 - s for s in similarities)
assert len(distances) == len(fps) * (len(fps) - 1) // 2

def group(cutoff):
    return Butina.ClusterData(distances, len(fps), distThresh=cutoff,
                              isDistData=True, reordering=True)

for cutoff in (0.25, 0.35, 0.45):
    clusters = group(cutoff)
    sizes = [len(c) for c in clusters]
    print("distance, clusters, singletons, largest:", cutoff, len(sizes), sizes.count(1), max(sizes))
clusters = group(0.35)
largest = max(clusters, key=len)
center = largest[0]
members = largest[1:]
def tc(i, j):
    return DataStructs.TanimotoSimilarity(fps[i], fps[j])
pair_score, a, b = min((tc(i, j), i, j) for n, i in enumerate(members) for j in members[n+1:])
print("structures, distances:", len(mols), len(distances))
for i in (center, a, b):
    print("index, SMILES:", i, library[i])
print("Tc(center,A), Tc(center,B), Tc(A,B):", tc(center, a), tc(center, b), pair_score)
image = Draw.MolsToGridImage([mols[i] for i in (a, center, b)], molsPerRow=3,
    subImgSize=(370, 285), legends=[f"A to center: {tc(center, a):.3f}", "Butina center", f"B to center: {tc(center, b):.3f}"])
image.save(figure_dir / "06-06-butina-clustering.png")


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-06-butina-clustering.png")))